# Phase B: real IBM Quantum hardware (Deliverable 3)

Before the first run: on IBM Quantum Platform create an API key and an instance (the Open Plan is free) and copy the instance CRN. Open Plan jobs run in job mode (no `Session`), which is what this notebook uses.

* `semiclassical` mode needs a QPU that supports dynamic circuits (mid-circuit measurement, reset, `if_else`). The backend search below only considers such QPUs. If none is available to you, set `MODE = "full"`.
* Keep `USE_TWIRL = False` in `semiclassical` mode: twirling may be rejected for dynamic circuits.
* **Coming back later with a fresh kernel?** Run the first two code cells and the helpers cell, then step 4. The job id is read from `results/ibm_job_<tag>.json`.

In [31]:
RUN_CORE_DEMO = False                  # load the core functions only, skip its demo/self-test cells
%run shor_core_generalized.ipynb

N, A = 15, 7                           # any odd composite N (not a prime power) and 1 < A < N, gcd(A, N) = 1
T = None                               # counting precision t; None = 2n
MODE = "semiclassical"                 # "semiclassical" = Beauregard 2n+3 qubits; "full" = t-qubit counting register

qc = build_shor_circuit(N, A, T, MODE, SKIP_IDENTITY)
t = qc.num_clbits
tag = run_tag(N, A, t, MODE)
print(f"{qc.name}: {qc.num_qubits} qubits, t = {t}")

shor_N15_a7_t8_semiclassical: 11 qubits, t = 8


In [32]:
import time
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

SHOTS = 4096
USE_DD = False          # dynamical decoupling
USE_TWIRL = False       # Pauli twirling (full mode only)
TRANSPILE_SEEDS = range(3)
POLL_SECONDS = 30

## 0. One-time: save your credentials

Set `SAVE_ACCOUNT = True`, run once, set it back to `False`. `getpass` keeps the key out of the notebook file. Credentials are stored in plain text in `~/.qiskit/qiskit-ibm.json`, so only do this on your own machine.

In [33]:
SAVE_ACCOUNT = True
if SAVE_ACCOUNT:
    from getpass import getpass
    QiskitRuntimeService.save_account(channel="ibm_quantum_platform",
                                      token=getpass("IBM Quantum API key: "),
                                      instance=input("Instance CRN: ").strip(),
                                      overwrite=True, set_as_default=True)
    print("Saved.")

Saved.


## Helpers: queue polling and result extraction

In [34]:
FINAL_STATES = {"DONE", "ERROR", "CANCELLED"}


def status_name(job):
    s = job.status()
    return getattr(s, "name", str(s))          # str or enum, depending on the runtime version


def wait_for(job):
    """Poll the queue, printing each status change. Interrupt any time; step 4 resumes from the saved id."""
    last = None
    while True:
        s = status_name(job)
        if s != last:
            print(f"[{time.strftime('%H:%M:%S')}] job {job.job_id()}: {s}")
            last = s
        if s in FINAL_STATES:
            return s
        time.sleep(POLL_SECONDS)


def save_counts(job, meta):
    pub_result = job.result()[0]                    # one circuit -> one PUB result
    counts = pub_result.data.c.get_counts()         # 'c' is the classical register in the core
    ideal = exact_distribution(meta["a"], meta["N"], meta["t"])
    print(f"Hellinger fidelity vs ideal: {hellinger_fidelity(ideal, counts):.4f}")
    print(f"Factoring success rate     : {success_rate(counts, meta['t'], meta['a'], meta['N']):.4f}")
    name = f"ibm_counts_{run_tag(meta['N'], meta['a'], meta['t'], meta['mode'])}.json"
    save_json(name, {**meta, "counts": counts})
    print(f"Saved {results_path(name)}")
    return counts

## 1. Least busy real QPU with enough qubits (and dynamic-circuit support in semiclassical mode)

Same rule as `service.least_busy`: the fewest pending jobs among the candidates.

In [35]:
service = QiskitRuntimeService()
candidates = service.backends(simulator=False, operational=True, min_num_qubits=qc.num_qubits)
if MODE == "semiclassical":
    candidates = [bk for bk in candidates if "if_else" in bk.target.operation_names]
assert candidates, "No suitable QPU available to this account (try MODE = 'full')"
backend = min(candidates, key=lambda bk: bk.status().pending_jobs)
print(f"Least busy QPU: {backend.name} ({backend.num_qubits} qubits, "
      f"{backend.status().pending_jobs} pending jobs)")

Least busy QPU: ibm_kingston (156 qubits, 0 pending jobs)


## 2. Transpile for that chip

In [36]:
isa, seed = best_isa_circuit(qc, backend, TRANSPILE_SEEDS)
stats = circuit_report(isa)
print(f"ISA: depth {stats['depth']}, two-qubit gates {stats['two_qubit_gates']}  (seed {seed})")

ISA: depth 13924, two-qubit gates 6401  (seed 0)


## 3. Submit with the Sampler primitive (once per experiment)

In [37]:
sampler = Sampler(mode=backend)
sampler.options.default_shots = SHOTS
if USE_DD:
    sampler.options.dynamical_decoupling.enable = True
if USE_TWIRL:
    if MODE == "semiclassical":
        print("Warning: twirling may be rejected for dynamic circuits")
    sampler.options.twirling.enable_gates = True
    sampler.options.twirling.enable_measure = True

job = sampler.run([isa])
meta = {"backend": backend.name, "job_id": job.job_id(), "N": N, "a": A, "t": t, "mode": MODE,
        "shots": SHOTS, "seed_transpiler": seed, "dd": USE_DD, "twirl": USE_TWIRL, "isa_stats": stats}
save_json(f"ibm_job_{tag}.json", meta)
print(f"Submitted job {job.job_id()}; id saved to {results_path(f'ibm_job_{tag}.json')}")

/tmp/ipykernel_25600/712561211.py:1: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = Sampler(mode=backend)


Submitted job dbipg9g1h92s738401e0; id saved to results/ibm_job_N15_a7_t8_semiclassical.json


## 4. Wait in the queue and fetch the counts

Works right after step 3, or later in a fresh kernel.

In [38]:
meta = load_json(f"ibm_job_{tag}.json")
assert meta is not None, "No saved job id: run step 3 first"
job = QiskitRuntimeService().job(meta["job_id"])
if wait_for(job) == "DONE":
    counts = save_counts(job, meta)
else:
    print("Job did not finish:", job.error_message())

[07:26:36] job dbipg9g1h92s738401e0: RUNNING
[07:27:06] job dbipg9g1h92s738401e0: DONE
Hellinger fidelity vs ideal: 0.9059
Factoring success rate     : 0.4709
Saved results/ibm_counts_N15_a7_t8_semiclassical.json
